In [1]:
from fastai.vision.all import *
import pandas as pd, json
from functools import partial
import torch

# Enable cuDNN auto‑tuner to select the fastest convolution algorithms
torch.backends.cudnn.benchmark = True




/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
path = Path("/kaggle/input/cassava-leaf-disease-classification/")




In [3]:
train_df = pd.read_csv(path / "train.csv")
with open(path / "label_num_to_disease_map.json") as f:
    label_dict = json.load(f)
label_dict = {int(k): v for k, v in label_dict.items()}

df = train_df.set_index("image_id")
labels = df.to_dict()["label"]


def get_label(labels, x):
    return labels[Path(x).name]


def get_data(labels, bs=64, presize=500, resize=384):
    tfms = [
        Rotate(90),
        Warp(magnitude=0.4, p=1.0),
        Zoom(min_zoom=0.9, max_zoom=1.3),
        Brightness(max_lighting=0.5),
        Flip(),
        Contrast(),
        Resize(resize),
    ]
    comp = setup_aug_tfms(tfms)
    return DataBlock(
        blocks=(ImageBlock, CategoryBlock),
        get_items=lambda p: get_image_files(p),
        get_y=partial(get_label, labels),
        splitter=RandomSplitter(),
        item_tfms=[Resize(presize)],
        batch_tfms=[*comp, Normalize.from_stats(*imagenet_stats)],
    ).dataloaders(path / "train_images", bs=bs, num_workers=8)




In [4]:
dls = get_data(labels, bs=128, presize=384)




In [5]:
# Use mixed‑precision (fp16) to accelerate training while keeping the same architecture and schedule
learn = cnn_learner(dls, models.resnet50, metrics=accuracy, pretrained=True).to_fp16()
learn.fine_tune(5, base_lr=1e-3)  # unchanged number of epochs and learning rate




/usr/local/lib/python3.11/dist-packages/fastai/vision/learner.py:303: UserWarning: `cnn_learner` has been renamed to `vision_learner` -- please update your code
  warn("`cnn_learner` has been renamed to `vision_learner` -- please update your code")
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 114MB/s]


epoch,train_loss,valid_loss,accuracy,time
0,1.433936,0.880473,0.724893,00:58


epoch,train_loss,valid_loss,accuracy,time
0,0.898852,0.730111,0.773504,01:05
1,0.737595,0.593305,0.815171,01:03
2,0.609710,0.502222,0.837340,01:03
3,0.561653,0.473384,0.848825,01:03
4,0.500412,0.472950,0.844551,01:03


In [6]:
test_files = get_image_files(path / "test_images")
test_dl = dls.test_dl(test_files)
tta_preds, _ = learn.tta(dl=test_dl, n=5)




In [7]:
preds = torch.argmax(tta_preds, dim=1).cpu().numpy()
test_ids = [f.name for f in test_files]




In [8]:
submission = pd.DataFrame({"image_id": test_ids, "label": preds})
submission.to_csv("submission.csv", index=False)
print("Submission file saved as submission.csv")

Submission file saved as submission.csv
